<a href="https://colab.research.google.com/github/IrineuBovoJunior398/Pos-em-IA/blob/main/TODAS_MEDIDAS_ESTAT%C3%8DSTICAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 🧠 Código Python completo para medidas estatísticas


import math
from collections import Counter
from statistics import mean, median, multimode, pstdev, pvariance, stdev, variance
from typing import List, Dict, Any, Optional
import numpy as np
from scipy import stats


def media_simples(dados: List[float]) -> float:
    return mean(dados)


def media_geometrica(dados: List[float]) -> float:
    # A média geométrica exige todos os valores > 0
    dados_array = np.array(dados, dtype=float)
    if np.any(dados_array <= 0):
        raise ValueError("Média geométrica só é definida para valores estritamente positivos.")
    return stats.gmean(dados_array)


def media_harmonica(dados: List[float]) -> float:
    dados_array = np.array(dados, dtype=float)
    if np.any(dados_array == 0):
        raise ValueError("Média harmônica não é definida quando há valores iguais a zero.")
    return stats.hmean(dados_array)


def media_ponderada(dados: List[float], pesos: Optional[List[float]] = None) -> float:
    if pesos is None:
        return mean(dados)
    if len(dados) != len(pesos):
        raise ValueError("dados e pesos precisam ter o mesmo tamanho.")
    dados_array = np.array(dados, dtype=float)
    pesos_array = np.array(pesos, dtype=float)
    return np.average(dados_array, weights=pesos_array)


def moda(dados: List[float]) -> List[float]:
    # multimode retorna todas as modas
    return multimode(dados)


def mediana(dados: List[float]) -> float:
    return median(dados)


def amplitude(dados: List[float]) -> float:
    return max(dados) - min(dados)


def quartis(dados: List[float]) -> Dict[str, float]:
    dados_array = np.array(dados, dtype=float)
    q1 = np.percentile(dados_array, 25)
    q2 = np.percentile(dados_array, 50)  # mediana
    q3 = np.percentile(dados_array, 75)
    return {"Q1": q1, "Q2": q2, "Q3": q3, "IQR": q3 - q1}


def decis(dados: List[float]) -> Dict[str, float]:
    dados_array = np.array(dados, dtype=float)
    d = {}
    for i in range(1, 10):
        d[f"D{i}"] = np.percentile(dados_array, i * 10)
    return d


def percentis(dados: List[float], step: int = 5) -> Dict[str, float]:
    """
    step=5 -> P5, P10, P15, ..., P95
    step=1 -> P1, P2, ..., P99
    """
    dados_array = np.array(dados, dtype=float)
    p = {}
    for i in range(step, 100, step):
        p[f"P{i}"] = np.percentile(dados_array, i)
    return p


def variancia_populacional(dados: List[float]) -> float:
    return pvariance(dados)


def variancia_amostral(dados: List[float]) -> float:
    return variance(dados)


def desvio_padrao_populacional(dados: List[float]) -> float:
    return pstdev(dados)


def desvio_padrao_amostral(dados: List[float]) -> float:
    return stdev(dados)


def coeficiente_variacao(dados: List[float]) -> float:
    m = mean(dados)
    if m == 0:
        raise ZeroDivisionError("Coeficiente de variação não definido para média igual a zero.")
    return stdev(dados) / m


def assimetria(dados: List[float]) -> float:
    # skewness (assimetria de Pearson)
    dados_array = np.array(dados, dtype=float)
    return stats.skew(dados_array, bias=False)


def curtose(dados: List[float]) -> float:
    # stats.kurtosis retorna a curtose "excesso" (0 = normal)
    dados_array = np.array(dados, dtype=float)
    return stats.kurtosis(dados_array, bias=False)


def frequencias(dados: List[Any]) -> Dict[Any, int]:
    return dict(Counter(dados))


def resumo_estatistico(
    dados: List[float],
    pesos: Optional[List[float]] = None,
    calcular_percentis: bool = False,
    passo_percentis: int = 5
) -> Dict[str, Any]:
    """
#    Retorna um dicionário com um resumo estatístico bem completo.
    """

    if len(dados) == 0:
        raise ValueError("A lista de dados não pode estar vazia.")

    dados_array = np.array(dados, dtype=float)

    resumo: Dict[str, Any] = {}

    # Informações básicas
    resumo["contagem"] = len(dados_array)
    resumo["min"] = float(np.min(dados_array))
    resumo["max"] = float(np.max(dados_array))
    resumo["soma"] = float(np.sum(dados_array))
    resumo["valores_unicos"] = len(np.unique(dados_array))

    # Tendência central
    resumo["media"] = media_simples(dados_array)
    resumo["mediana"] = mediana(dados_array)
    resumo["moda"] = moda(dados_array)

    # Médias especiais (com try para evitar erro em dados incompatíveis)
    try:
        resumo["media_geometrica"] = media_geometrica(dados_array)
    except ValueError as e:
        resumo["media_geometrica"] = str(e)

    try:
        resumo["media_harmonica"] = media_harmonica(dados_array)
    except ValueError as e:
        resumo["media_harmonica"] = str(e)

    resumo["media_ponderada"] = media_ponderada(dados_array, pesos)

    # Dispersão
    resumo["amplitude"] = amplitude(dados_array)
    resumo["variancia_populacional"] = variancia_populacional(dados_array)
    resumo["variancia_amostral"] = variancia_amostral(dados_array)
    resumo["desvio_padrao_populacional"] = desvio_padrao_populacional(dados_array)
    resumo["desvio_padrao_amostral"] = desvio_padrao_amostral(dados_array)

    try:
        resumo["coeficiente_variacao"] = coeficiente_variacao(dados_array)
    except ZeroDivisionError as e:
        resumo["coeficiente_variacao"] = str(e)

    # Posição
    resumo["quartis"] = quartis(dados_array)
    resumo["decis"] = decis(dados_array)

    if calcular_percentis:
        resumo["percentis"] = percentis(dados_array, step=passo_percentis)

    # Forma
    resumo["assimetria"] = assimetria(dados_array)
    resumo["curtose_excesso"] = curtose(dados_array)

    # Frequências
    resumo["frequencias"] = frequencias(dados_array)

    return resumo


# Exemplo de uso
if __name__ == "__main__":
    # Exemplo simples de dados
    dados_exemplo = [1, 2, 2, 3, 4, 5, 5, 5, 10]

    # opcional: pesos para média ponderada
    pesos_exemplo = [1, 1, 1, 1, 1, 1, 1, 1, 2]

    resultado = resumo_estatistico(
        dados_exemplo,
        pesos=pesos_exemplo,
        calcular_percentis=True,
        passo_percentis=10  # P10, P20, ..., P90
    )

    # Impressão organizada
    import pprint
    pprint.pp(resultados := resultado)  # só para exibir bonito

{'contagem': 9,
 'min': 1.0,
 'max': 10.0,
 'soma': 37.0,
 'valores_unicos': 6,
 'media': np.float64(4.111111111111111),
 'mediana': np.float64(4.0),
 'moda': [np.float64(5.0)],
 'media_geometrica': np.float64(3.395515321684607),
 'media_harmonica': np.float64(2.7411167512690358),
 'media_ponderada': np.float64(4.7),
 'amplitude': np.float64(9.0),
 'variancia_populacional': np.float64(6.320987654320987),
 'variancia_amostral': np.float64(7.111111111111111),
 'desvio_padrao_populacional': 2.5141574442188355,
 'desvio_padrao_amostral': 2.6666666666666665,
 'coeficiente_variacao': np.float64(0.6486486486486487),
 'quartis': {'Q1': np.float64(2.0),
             'Q2': np.float64(4.0),
             'Q3': np.float64(5.0),
             'IQR': np.float64(3.0)},
 'decis': {'D1': np.float64(1.8),
           'D2': np.float64(2.0),
           'D3': np.float64(2.4),
           'D4': np.float64(3.2),
           'D5': np.float64(4.0),
           'D6': np.float64(4.8),
           'D7': np.float64(5.0),